# Import libraries

In [1]:
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearnex import patch_sklearn
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from statsmodels.graphics.tsaplots import plot_acf

patch_sklearn()
np.random.seed(2026)

# Visual style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

ModuleNotFoundError: No module named 'sklearnex'

# Load data

**Variable and Parameters Names**
 - Climate = the original data set (146 observations with 19 parameters)
 - Months = Monthly climate data
 - J-D = Jan - Dec (Yearly temp)
 - N-D = Dec - Nov (Meteorological Year)
 - DJF = Dec, Jan, Feb
 - MAM = Mar, Apr, May
 - JJA = June, July, Aug
 - SON = Sep, Oct, Nov

In [ ]:
df = pd.read_csv("data/GLB.Ts+dSST.csv", skiprows=1, na_values="***")

month_cols = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

data = df[['Year'] + month_cols].copy()
data = data.dropna()

data.head()

In [ ]:
# Melt the data from wide to long format
data = data.melt(id_vars='Year', var_name='Month', value_name='Anomaly')

# Datetime index
data['Date'] = pd.to_datetime(data['Year'].astype(str) + '-' + data['Month'] + '-01')

# Set index and frequency
data = data.sort_values('Date').set_index('Date')
data.index.freq = 'MS' # 'MS' = 'Month Start'

data.head()

# Initial visualization

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Date', y='Anomaly', data=data, color='maroon', label='Observed Data', s=10)

# 5-year rolling average
data['Rolling_Avg'] = data['Anomaly'].rolling(window=5).mean()
sns.lineplot(x='Date', y='Rolling_Avg', data=data, color='black', label='12-Month Moving Average', alpha=0.6)

plt.title('Global Temperature Anomalies (1880-2025)', fontsize=14)
plt.ylabel('Temperature Anomaly (°C, 1951-1980 Baseline)')
plt.xlabel('Year')
plt.legend(loc='upper left')
plt.savefig('plots\\observed_data.png', dpi=100, bbox_inches='tight', transparent=True)
plt.show()

# Random Forest Regression

In [ ]:
df_lag = data[['Year', 'Anomaly']].copy()
lags = 24

# Create lag features
for l in range(1, lags+1):
    df_lag[f'Lag_{l}'] = df_lag['Anomaly'].shift(l)

# Drop NaN rows created by lags
df_lag = df_lag.dropna()

df_lag.head()

In [ ]:
X = df_lag[[f'Lag_{lag}' for lag in range(1, lags+1)]]
y = df_lag['Anomaly']

In [ ]:
%%time

# Time-Series split
tscv = TimeSeriesSplit(n_splits=5, test_size=24)

# Random forest model
rf_model = RandomForestRegressor(random_state=2026)

# Define parameters for tuning
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [1, 'sqrt', 'log2', None],
}

# Perform Grid Search
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)

grid_search.fit(X, y)

# Best model
best_rf = grid_search.best_estimator_
print(f"Best parameters: {grid_search.best_params_}")

In [ ]:
# MSE
y_pred_rf = best_rf.predict(X)
mse_rf = mean_squared_error(y, y_pred_rf)

print(f"Random Forest MSE: {mse_rf:.5f}")

# Forecast

In [ ]:
%%time

# Generate a recursive forecast
forecast_period = 120
rf_forecast = []

# Latest observed data
current_lags = X.iloc[-1:].copy()

for _ in range(forecast_period):
    next_pred = best_rf.predict(current_lags)[0]
    rf_forecast.append(next_pred)
    
    # Update lags
    for lag in range(lags, 1, -1):
        current_lags[f'Lag_{lag}'] = current_lags[f'Lag_{lag-1}']

    current_lags['Lag_1'] = next_pred

## Visualize forecast

In [ ]:
# Date limits
start_date = '2000-01-01'
latest_date = data.index.max()
end_date = latest_date + pd.DateOffset(months=forecast_period)

plt.figure(figsize=(12, 6))

# Observed data
#plt.plot(data.index, data['Anomaly'], label='Observed Data', color='black', alpha=0.6)
plt.scatter(data.index, data['Anomaly'], label='Observed Data', color='maroon', alpha=0.3)

# RF fit
plt.plot(df_lag.index, y_pred_rf, label='RF Fit', color='green', linestyle='--')

# Forecast
forecast_series = pd.date_range(start=latest_date + pd.DateOffset(months=1), end=end_date, freq='MS')
plt.plot(forecast_series, rf_forecast, label=f'RF {forecast_period}-Month Forecast', color='red', linestyle='--')

# Set x-axis limits
plt.xlim(pd.Timestamp(start_date), end_date + pd.DateOffset(years=1))

plt.title(f'Random Forest Fit with {forecast_period}-Month Forecast')
plt.xlabel('Year')
plt.ylabel('Temperature Anomaly (°C)')
plt.legend(loc='upper left')
#plt.savefig('plots\\random_forest_fit.png', dpi=100, bbox_inches='tight', transparent=True)
plt.savefig('plots\\random_forest_fit.png', dpi=100, bbox_inches='tight')
plt.show()

# Bootstrap

In [ ]:
%%time

# Residuals from best model
residuals = y - best_rf.predict(X)

# Bootstrap settings
n_bootstraps = 100
all_boot_forecasts = []

# Run bootstrap
for i in range(n_bootstraps):
    boot_forecast = []
    current_lags = X.iloc[-1:].copy()
    
    for _ in range(forecast_period):
        pred = best_rf.predict(current_lags).item()
        
        # Add random error from residuals
        random_error = np.random.choice(residuals)
        noisy_pred = pred + random_error
        
        boot_forecast.append(noisy_pred)
        
        # Update lags
        for lag in range(lags, 1, -1):
            current_lags[f'Lag_{lag}'] = current_lags[f'Lag_{lag-1}']
        current_lags['Lag_1'] = noisy_pred
        
    all_boot_forecasts.append(boot_forecast)

# Calculate the 95% Confidence Interval
all_boot_forecasts = np.array(all_boot_forecasts)
lower_bound = np.percentile(all_boot_forecasts, 2.5, axis=0)
upper_bound = np.percentile(all_boot_forecasts, 97.5, axis=0)
mean_forecast = np.mean(all_boot_forecasts, axis=0)

## Visualize Bootstrap Confidence Interval

In [ ]:
plt.figure(figsize=(12, 6))

# Observed data
#plt.plot(data.index, data['Anomaly'], label='Observed Data', color='black', alpha=0.6)
plt.scatter(data.index, data['Anomaly'], label='Observed Data', color='maroon', alpha=0.3)

# RF fit
plt.plot(df_lag.index, y_pred_rf, label='RF Fit', color='green', linestyle='--')

# Forecast
plt.plot(forecast_series, mean_forecast, label='RF Bootstrap Mean', color='red', linestyle='--')

# 95% Confidence Interval
plt.fill_between(forecast_series, lower_bound, upper_bound, color='red', alpha=0.2, label='95% Bootstrap CI')

# Set x-axis limits
plt.xlim(pd.Timestamp(start_date), end_date + pd.DateOffset(years=1))

plt.title(f'Random Forest Fit with {forecast_period}-Year Bootstrap Forecast', fontsize=14)
plt.xlabel('Year')
plt.ylabel('Temperature Anomaly (°C)')
plt.legend(loc='upper left')
#plt.savefig('plots/random_forest_forecast.png', dpi=100, bbox_inches='tight', transparent=True)
plt.savefig('plots/random_forest_forecast.png', dpi=100, bbox_inches='tight')
plt.show()

# Residual Diagnostics

In [ ]:
def run_residual_diagnostics(x_index, y_actual, y_pred, model_name, plt_name=False):
    # Calculate residuals
    residuals = y_actual - y_pred
    
    # Initialize figure with 3 subplots
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Residual Diagnostics: {model_name}', fontsize=14)

    # Residuals vs Time
    axes[0].scatter(x_index, residuals, color='maroon', alpha=0.3)
    axes[0].axhline(0, color='black', linestyle='--')
    axes[0].set_title('Residuals vs. Time')
    axes[0].set_ylabel('Error (°C)')
    axes[0].set_xlabel('Year')

    # Normality check
    sns.histplot(residuals, kde=True, ax=axes[1], color='grey')
    axes[1].set_title('Distribution of Residuals')
    axes[1].set_xlabel('Error (°C)')
    axes[1].set_ylabel('Frequency')
    
    # ACF
    plot_acf(residuals, ax=axes[2], lags=20, color='maroon', alpha=0.3)
    axes[2].set_title('Autocorrelation of Residuals')
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    if (plt_name):
        #plt.savefig(plt_name, dpi=100, bbox_inches='tight', transparent=True)
        plt.savefig(plt_name, dpi=100, bbox_inches='tight')
    plt.show()

In [ ]:
run_residual_diagnostics(
    df_lag.index,
    y,
    y_pred_rf,
    f'Random Forest (Lags={lags})',
    plt_name='plots/rf_residual_diagnostics.png'
)

# Feature Importance

In [ ]:
importances = best_rf.feature_importances_
feature_names = X.columns.tolist()

plt.figure(figsize=(10, 6))
plt.barh(feature_names, importances, color='maroon', alpha=0.6)
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance Score')
#plt.savefig('plots\\random_forest_feature_importance.png', dpi=100, bbox_inches='tight', transparent=True)
plt.savefig('plots\\random_forest_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

# Result comparison

In [ ]:
holdout_period = 120

X_train = X.iloc[:-holdout_period]
y_train = y.iloc[:-holdout_period]
X_test  = X.iloc[-holdout_period:]
y_test  = y.iloc[-holdout_period:]

print(f"Training years: {df_lag['Year'].iloc[0]} to {df_lag['Year'].iloc[-holdout_period-1]}")
print(f"Holdout years: {df_lag['Year'].iloc[-holdout_period]} to {df_lag['Year'].iloc[-1]}")

In [ ]:
tscv = TimeSeriesSplit(n_splits=5, test_size=24)
cv_errors = []

def recursive_rf_forecast(model, history_values, future_index, lags=24):
    history_values = list(np.asarray(history_values, dtype=float))
    preds = []

    for current_date in future_index:
        row = {f'Lag_{lag}': history_values[-lag] for lag in range(1, lags + 1)}
        X_next = pd.DataFrame([row])
        yhat = float(model.predict(X_next)[0])
        preds.append(yhat)
        history_values.append(yhat)

    return pd.Series(preds, index=future_index)

for fold, (train_index, val_index) in enumerate(tscv.split(y_train), 1):
    y_cv_train = y_train.iloc[train_index]
    y_cv_val = y_train.iloc[val_index]

    # rebuild lagged training matrix using only fold training data
    fold_train_df = pd.DataFrame({"Anomaly": y_cv_train})
    for l in range(1, lags + 1):
        fold_train_df[f'Lag_{l}'] = fold_train_df['Anomaly'].shift(l)
    fold_train_df = fold_train_df.dropna()

    X_cv_train = fold_train_df[[f'Lag_{lag}' for lag in range(1, lags + 1)]]
    y_cv_train_fit = fold_train_df['Anomaly']

    best_rf.fit(X_cv_train, y_cv_train_fit)

    cv_preds = recursive_rf_forecast(
        model=best_rf,
        history_values=y_cv_train.values,
        future_index=y_cv_val.index,
        lags=lags
    )

    mse = mean_squared_error(y_cv_val, cv_preds)
    cv_errors.append(float(mse))

    print(f"Fold {fold} MSE: {mse:.5f}")

print("-"*25)
print(f"CV errors: {cv_errors}")
print(f"Mean CV MSE: {np.mean(cv_errors):.5f}")

In [ ]:
best_rf.fit(X_train, y_train)

y_pred = recursive_rf_forecast(
    model=best_rf,
    history_values=y_train.values,
    future_index=y_test.index,
    lags=lags
)

results = {
    "model_name": "Random Forest",
    "cv_metric_name": "MSE",
    "cv_errors": cv_errors,
    "holdout_actual": y_test.tolist(),
    "holdout_pred": y_pred.tolist()
}

with open("random_forest_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved random_forest_results.json")
print("Holdout length actual:", len(results["holdout_actual"]))
print("Holdout length pred:", len(results["holdout_pred"]))

In [ ]:
comparison_df = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred,
    'Error': y_test - y_pred
}, index=X_test.index)

print(f"\n{'='*50}")
print(f"Model: {results['model_name']}")
print(f"CV Metric: {results['cv_metric_name']}")
print(f"CV Errors (mean ± std): {np.mean(results['cv_errors']):.4f} ± {np.std(results['cv_errors']):.4f}")
print(f"{'='*50}")

with pd.option_context('display.max_rows', None, 'display.width', None):
    print(comparison_df.round(4))